In [ ]:
import numpy as np
from datetime import datetime, timedelta

FILE_PATH = "/WhatsApp Chat with RVITM BATCH 7.txt"

print("GroupDNA loaded successfully.")

GroupDNA loaded successfully.


In [ ]:
with open(FILE_PATH, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Chat file loaded successfully.")
print("Total lines:", len(lines))

Chat file loaded successfully.
Total lines: 4029


In [ ]:
def convert_timestamp(timestamp_text):

    formats = [
        "%d/%m/%Y, %H:%M",
        "%d/%m/%y, %H:%M"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(timestamp_text, fmt)
        except ValueError:
            continue

    return None


def is_message_start(line):

    if " - " not in line:
        return False

    timestamp_text = line.split(" - ", 1)[0]

    return convert_timestamp(timestamp_text) is not None

In [ ]:
def parse_chat(lines):

    records = []
    system_messages = 0
    current_message = None

    for raw_line in lines:

        line = raw_line.rstrip("\n").strip()

        # Ignore empty lines
        if line == "":
            continue

        # Check for a new timestamped message
        if is_message_start(line):

            timestamp_text, rest = line.split(" - ", 1)

            timestamp = convert_timestamp(timestamp_text)

            # Normal user message
            if ": " in rest:

                sender, text = rest.split(": ", 1)

                sender = sender.strip()
                text = text.strip()

                current_message = {
                    "timestamp": timestamp_text,
                    "datetime": timestamp,
                    "sender": sender,
                    "text": text,
                    "is_media": text == "<Media omitted>",
                    "is_deleted": text == "This message was deleted"
                }

                records.append(current_message)

            # System message
            else:

                system_messages += 1
                current_message = None

        # Multiline continuation
        else:

            if current_message is not None:
                current_message["text"] += "\n" + line

    return records, system_messages

In [ ]:
records, system_messages = parse_chat(lines)

# Make sure all messages are chronologically ordered
records.sort(key=lambda message: message["datetime"])

messages = records

print("Messages parsed:", len(messages))
print("System messages:", system_messages)

Messages parsed: 1094
System messages: 497


In [ ]:
text_messages = []
media_messages = []
deleted_messages = []

for message in messages:

    if message["is_media"]:
        media_messages.append(message)

    elif message["is_deleted"]:
        deleted_messages.append(message)

    else:
        text_messages.append(message)

print("Text messages   :", len(text_messages))
print("Media messages  :", len(media_messages))
print("Deleted messages:", len(deleted_messages))

Text messages   : 564
Media messages  : 328
Deleted messages: 202


In [ ]:
participants = set()

for message in messages:
    participants.add(message["sender"])

participants = sorted(participants)

print("Active participants:", len(participants))

for person in participants:
    print("-", person)

Active participants: 179
- +91 62006 99712
- +91 62032 75940
- +91 62043 12884
- +91 62810 10673
- +91 62915 15117
- +91 62943 57358
- +91 62997 56380
- +91 63611 68135
- +91 63612 16843
- +91 63613 63701
- +91 63613 71939
- +91 63613 88908
- +91 63631 45915
- +91 63634 60669
- +91 63645 50111
- +91 63645 79181
- +91 63646 74041
- +91 70040 65482
- +91 70047 69323
- +91 70196 27836
- +91 70199 98516
- +91 72260 60982
- +91 72595 06907
- +91 72600 66571
- +91 73490 65512
- +91 74069 57769
- +91 74110 69387
- +91 74113 23557
- +91 74116 90905
- +91 74149 74464
- +91 74835 29257
- +91 74990 10687
- +91 76191 15586
- +91 76196 00389
- +91 76450 31687
- +91 76728 60396
- +91 76762 75017
- +91 76762 77946
- +91 76763 67338
- +91 76767 33602
- +91 77606 39981
- +91 77607 66481
- +91 77804 60892
- +91 77999 34399
- +91 78922 56796
- +91 78990 08625
- +91 80502 71048
- +91 80732 94215
- +91 80884 40566
- +91 80951 40026
- +91 81054 14600
- +91 81055 75119
- +91 81233 90662
- +91 81236 20533
- +

In [ ]:
print("=" * 60)
print("PARSER VALIDATION")
print("=" * 60)

print("Total parsed messages :", len(messages))
print("Active participants   :", len(participants))
print("System messages       :", system_messages)
print("Media messages        :", len(media_messages))
print("Deleted messages      :", len(deleted_messages))

if messages:

    print(
        "First message         :",
        messages[0]["datetime"].strftime("%d/%m/%Y, %H:%M")
    )

    print(
        "Last message          :",
        messages[-1]["datetime"].strftime("%d/%m/%Y, %H:%M")
    )

print("=" * 60)

PARSER VALIDATION
Total parsed messages : 1094
Active participants   : 179
System messages       : 497
Media messages        : 328
Deleted messages      : 202
First message         : 02/09/2025, 11:56
Last message          : 19/09/2026, 21:01


In [ ]:
def group_overview(messages, participants):

    total_messages = len(messages)

    participant_stats = {}

    for person in participants:

        count = 0

        for message in messages:

            if message["sender"] == person:
                count += 1

        if total_messages > 0:
            percentage = (count / total_messages) * 100
        else:
            percentage = 0

        participant_stats[person] = {
            "messages": count,
            "percentage": percentage
        }

    if messages:

        first_message = messages[0]["datetime"]
        last_message = messages[-1]["datetime"]

        active_days = (
            last_message.date() - first_message.date()
        ).days + 1

    else:

        first_message = None
        last_message = None
        active_days = 0

    return {
        "total_messages": total_messages,
        "participants": len(participants),
        "first_message": first_message,
        "last_message": last_message,
        "active_days": active_days,
        "participant_stats": participant_stats
    }

In [ ]:
def busiest_day_and_hour(messages):

    day_counts = {}
    hour_counts = {}

    for message in messages:

        day = message["datetime"].date()
        hour = message["datetime"].hour

        if day not in day_counts:
            day_counts[day] = 0

        day_counts[day] += 1

        if hour not in hour_counts:
            hour_counts[hour] = 0

        hour_counts[hour] += 1

    if not day_counts:

        return {
            "day": None,
            "day_count": 0,
            "hour": 0,
            "hour_count": 0
        }

    busiest_day = max(
        day_counts,
        key=day_counts.get
    )

    busiest_hour = max(
        hour_counts,
        key=hour_counts.get
    )

    return {
        "day": busiest_day,
        "day_count": day_counts[busiest_day],
        "hour": busiest_hour,
        "hour_count": hour_counts[busiest_hour]
    }

In [ ]:
def build_activity_matrix(messages, participants):

    matrix = np.zeros(
        (len(participants), 24),
        dtype=int
    )

    participant_index = {}

    for index, person in enumerate(participants):
        participant_index[person] = index

    for message in messages:

        person = message["sender"]
        hour = message["datetime"].hour

        row = participant_index[person]

        matrix[row][hour] += 1

    return matrix

In [ ]:
def render_heatmap(matrix, participants):

    print("  ACTIVITY HEATMAP (hour of day, columns 00 to 23)")
    print()
    print("                 00 03 06 09 12 15 18 21")

    if matrix.size == 0:
        print("  No activity data available.")
        return

    max_value = np.max(matrix)

    if max_value == 0:
        max_value = 1

    for index, person in enumerate(participants):

        row_output = ""

        for hour in range(0, 24, 3):

            block_count = (
                matrix[index][hour]
                + matrix[index][hour + 1]
                + matrix[index][hour + 2]
            )

            ratio = block_count / max_value

            if block_count == 0:
                symbol = "·"

            elif ratio < 0.10:
                symbol = "░"

            elif ratio < 0.30:
                symbol = "▒"

            elif ratio < 0.60:
                symbol = "▓"

            else:
                symbol = "█"

            row_output += f" {symbol} "

        print(f"    {person[:18]:<18}{row_output}")

In [ ]:
STOP_WORDS = {
    "the", "is", "a", "an", "and", "or", "to", "of",
    "in", "on", "for", "with", "this", "that", "it",
    "i", "you", "we", "he", "she", "they", "me",
    "my", "your", "our", "are", "was", "were",
    "be", "been", "am", "at", "as", "from", "by",
    "have", "has", "had", "will", "can", "just",
    "but", "so", "if", "not", "do", "did", "dont",
    "don't", "yes", "no", "ok", "okay"
}


def clean_word(word):

    word = word.lower()

    punctuation = ".,!?;:()[]{}<>/\\\"'|-_…"

    for symbol in punctuation:
        word = word.replace(symbol, "")

    return word

In [ ]:
def get_word_frequency(text_messages):

    word_frequency = {}

    for message in text_messages:

        words = message["text"].split()

        for raw_word in words:

            word = clean_word(raw_word)

            if word == "":
                continue

            if word in STOP_WORDS:
                continue

            # Ignore links
            if word.startswith("http") or "www" in word:
                continue

            has_letter = False

            for character in word:

                if character.isalpha():
                    has_letter = True
                    break

            if not has_letter:
                continue

            if word not in word_frequency:
                word_frequency[word] = 0

            word_frequency[word] += 1

    return word_frequency

In [ ]:
def message_length_stats(text_messages):

    if len(text_messages) == 0:

        return {
            "average": 0,
            "longest": 0
        }

    total_words = 0
    longest = 0

    for message in text_messages:

        length = len(
            message["text"].split()
        )

        total_words += length

        if length > longest:
            longest = length

    average = total_words / len(text_messages)

    return {
        "average": average,
        "longest": longest
    }

In [ ]:
def calculate_response_times(messages):

    response_times = {}

    for person in participants:
        response_times[person] = []

    for index in range(1, len(messages)):

        previous_message = messages[index - 1]
        current_message = messages[index]

        previous_sender = previous_message["sender"]
        current_sender = current_message["sender"]

        # A response is counted only when the sender changes
        if previous_sender != current_sender:

            gap = (
                current_message["datetime"]
                - previous_message["datetime"]
            )

            minutes = gap.total_seconds() / 60

            if minutes >= 0:
                response_times[current_sender].append(
                    minutes
                )

    return response_times

In [ ]:
def calculate_silent_streaks(messages, participants):

    if not messages:

        return {
            person: {
                "days": 0,
                "start": None,
                "end": None
            }
            for person in participants
        }

    first_date = messages[0]["datetime"].date()
    last_date = messages[-1]["datetime"].date()

    active_dates = {}

    for person in participants:
        active_dates[person] = set()

    for message in messages:

        person = message["sender"]
        date = message["datetime"].date()

        active_dates[person].add(date)

    results = {}

    current_date = first_date

    all_dates = []

    while current_date <= last_date:

        all_dates.append(current_date)

        current_date += timedelta(days=1)

    for person in participants:

        longest_days = 0
        longest_start = None
        longest_end = None

        current_silent_days = 0
        current_start = None

        for date in all_dates:

            if date not in active_dates[person]:

                if current_silent_days == 0:
                    current_start = date

                current_silent_days += 1

                if current_silent_days > longest_days:

                    longest_days = current_silent_days
                    longest_start = current_start
                    longest_end = date

            else:

                current_silent_days = 0
                current_start = None

        results[person] = {
            "days": longest_days,
            "start": longest_start,
            "end": longest_end
        }

    return results

In [ ]:
def spammer_score(messages, person):

    longest_burst = 0
    current_burst = 0
    previous_sender = None

    for message in messages:

        sender = message["sender"]

        if sender == person:

            if previous_sender == person:
                current_burst += 1
            else:
                current_burst = 1

            if current_burst > longest_burst:
                longest_burst = current_burst

        else:

            current_burst = 0

        previous_sender = sender

    return longest_burst

In [ ]:
def group_mom_score(messages, person):

    caring_words = {
        "guys", "please", "remember", "submit",
        "deadline", "attendance", "food", "eat",
        "sleep", "careful", "bring", "dont",
        "don't", "class", "exam"
    }

    score = 0

    for message in messages:

        if message["sender"] != person:
            continue

        words = message["text"].lower().split()

        for word in words:

            cleaned = clean_word(word)

            if cleaned in caring_words:
                score += 1

    return score

In [ ]:
def night_owl_score(messages, person):

    total = 0
    night = 0

    for message in messages:

        if message["sender"] != person:
            continue

        total += 1

        hour = message["datetime"].hour

        if hour >= 23 or hour <= 4:
            night += 1

    if total == 0:
        return 0

    return (night / total) * 100

In [ ]:
def storyteller_score(messages, person):

    total_words = 0
    message_count = 0

    for message in messages:

        if message["sender"] != person:
            continue

        if message["is_media"] or message["is_deleted"]:
            continue

        total_words += len(
            message["text"].split()
        )

        message_count += 1

    if message_count == 0:
        return 0

    return total_words / message_count

In [ ]:
def drama_queen_score(messages, person):

    score = 0

    for message in messages:

        if message["sender"] != person:
            continue

        text = message["text"]

        score += text.count("!")
        score += text.count("?")

        uppercase_letters = 0
        total_letters = 0

        for character in text:

            if character.isalpha():

                total_letters += 1

                if character.isupper():
                    uppercase_letters += 1

        if total_letters > 0:

            uppercase_ratio = (
                uppercase_letters / total_letters
            )

            if uppercase_ratio > 0.60:
                score += 3

    return score

In [ ]:
def ghost_score(messages, person):

    active_dates = set()

    for message in messages:

        if message["sender"] == person:
            active_dates.add(
                message["datetime"].date()
            )

    if len(active_dates) < 2:
        return 0

    dates = sorted(active_dates)

    longest_gap = 0

    for index in range(1, len(dates)):

        gap = (
            dates[index] - dates[index - 1]
        ).days - 1

        if gap > longest_gap:
            longest_gap = gap

    return longest_gap

In [ ]:
def determine_archetypes(
    messages,
    participants,
    silent_streaks
):

    results = {}

    for person in participants:

        scores = {

            "THE SPAMMER":
                spammer_score(
                    messages,
                    person
                ),

            "THE GROUP MOM":
                group_mom_score(
                    messages,
                    person
                ),

            "THE NIGHT OWL":
                night_owl_score(
                    messages,
                    person
                ),

            "THE STORYTELLER":
                storyteller_score(
                    messages,
                    person
                ),

            "THE DRAMA QUEEN":
                drama_queen_score(
                    messages,
                    person
                ),

            "THE GHOST":
                silent_streaks[person]["days"]
        }

        results[person] = scores

    return results

In [ ]:
def choose_archetypes(
    archetype_scores,
    participants
):

    maximums = {}

    for archetype in archetype_scores[participants[0]]:

        maximums[archetype] = 0

        for person in participants:

            value = archetype_scores[person][archetype]

            if value > maximums[archetype]:
                maximums[archetype] = value

    final_archetypes = {}

    for person in participants:

        best_archetype = None
        best_score = -1

        for archetype in archetype_scores[person]:

            value = archetype_scores[person][archetype]
            maximum = maximums[archetype]

            if maximum == 0:
                normalized = 0
            else:
                normalized = value / maximum

            if normalized > best_score:

                best_score = normalized
                best_archetype = archetype

        final_archetypes[person] = {
            "archetype": best_archetype,
            "raw_score":
                archetype_scores[person][best_archetype],
            "normalized_score":
                best_score
        }

    return final_archetypes

In [ ]:
def archetype_detail(
    person,
    archetype,
    raw_score
):

    if archetype == "THE SPAMMER":

        return (
            f"(longest burst: "
            f"{int(raw_score)} msgs in a row)"
        )

    elif archetype == "THE GROUP MOM":

        return (
            f"(caring keyword score: "
            f"{int(raw_score)})"
        )

    elif archetype == "THE NIGHT OWL":

        return (
            f"({raw_score:.1f}% msgs "
            f"between 23h-04h)"
        )

    elif archetype == "THE STORYTELLER":

        return (
            f"(avg {raw_score:.1f} words)"
        )

    elif archetype == "THE DRAMA QUEEN":

        return (
            f"(reaction score: "
            f"{int(raw_score)})"
        )

    elif archetype == "THE GHOST":

        return (
            f"(silent for "
            f"{int(raw_score)} days)"
        )

    return ""


def display_archetypes(
    final_archetypes,
    participants
):

    for person in participants:

        data = final_archetypes[person]

        archetype = data["archetype"]
        raw_score = data["raw_score"]

        detail = archetype_detail(
            person,
            archetype,
            raw_score
        )

        print(
            f"    {person[:20]:<20} "
            f"→ {archetype:<18} "
            f"{detail}"
        )

In [ ]:
def print_bar(
    value,
    maximum,
    width=22
):

    if maximum <= 0:
        return ""

    bar_length = int(
        (value / maximum) * width
    )

    return "█" * bar_length


def final_report():

    overview = group_overview(
        messages,
        participants
    )

    busiest = busiest_day_and_hour(
        messages
    )

    activity_matrix = build_activity_matrix(
        messages,
        participants
    )

    word_frequency = get_word_frequency(
        text_messages
    )

    top_words = sorted(
        word_frequency.items(),
        key=lambda item: item[1],
        reverse=True
    )[:5]

    response_times = calculate_response_times(
        messages
    )

    silent_streaks = calculate_silent_streaks(
        messages,
        participants
    )

    archetype_scores = determine_archetypes(
        messages,
        participants,
        silent_streaks
    )

    final_archetypes = choose_archetypes(
        archetype_scores,
        participants
    )

    participant_stats = (
        overview["participant_stats"]
    )

    # ========================================================
    # HEADER
    # ========================================================

    print()
    print("=" * 60)

    print(
        'GROUPDNA REPORT — "RVITM BATCH 7"'
    )

    print(
        f"{overview['active_days']} days  •  "
        f"{overview['total_messages']:,} messages  •  "
        f"{overview['participants']} participants"
    )

    print("=" * 60)

    # ========================================================
    # PERIOD
    # ========================================================

    print()

    print(
        "  Period          : "
        f"{overview['first_message'].strftime('%d %B %Y')} "
        f"to "
        f"{overview['last_message'].strftime('%d %B %Y')}"
    )

    if busiest["day"] is not None:

        print(
            "  Busiest day     : "
            f"{busiest['day'].strftime('%d %B %Y')} "
            f"({busiest['day_count']} messages)"
        )

        end_hour = (
            busiest["hour"] + 1
        ) % 24

        print(
            "  Busiest hour    : "
            f"{busiest['hour']:02d}:00 - "
            f"{end_hour:02d}:00"
        )

    # ========================================================
    # MESSAGES PER PERSON
    # ========================================================

    print()
    print("  MESSAGES PER PERSON")

    maximum_messages = max(
        [
            stats["messages"]
            for stats in participant_stats.values()
        ],
        default=0
    )

    sorted_people = sorted(
        participant_stats.items(),
        key=lambda item: item[1]["messages"],
        reverse=True
    )

    for person, stats in sorted_people:

        bar = print_bar(
            stats["messages"],
            maximum_messages,
            width=22
        )

        print(
            f"    {person[:18]:<18} "
            f"{bar:<22} "
            f"{stats['messages']:>6} "
            f"({stats['percentage']:.1f}%)"
        )

    # ========================================================
    # ACTIVITY HEATMAP
    # ========================================================

    print()
    render_heatmap(
        activity_matrix,
        participants
    )

    # ========================================================
    # TOP WORDS
    # ========================================================

    print()
    print("  THIS GROUP'S FAVOURITE WORDS")

    maximum_word_count = (
        top_words[0][1]
        if top_words
        else 0
    )

    for word, count in top_words:

        bar = print_bar(
            count,
            maximum_word_count,
            width=20
        )

        print(
            f"    {word:<15}"
            f"{bar:<20}"
            f"{count}"
        )

    # ========================================================
    # RESPONSE PATTERNS
    # ========================================================

    print()
    print("  RESPONSE PATTERNS")

    average_response = {}

    for person in participants:

        if response_times[person]:

            average_response[person] = (
                sum(response_times[person])
                / len(response_times[person])
            )

    if average_response:

        fastest_person = min(
            average_response,
            key=average_response.get
        )

        slowest_person = max(
            average_response,
            key=average_response.get
        )

        fastest_time = (
            average_response[fastest_person]
        )

        slowest_time = (
            average_response[slowest_person]
        )

        print(
            f"    Fastest replier : "
            f"{fastest_person} "
            f"(avg {fastest_time:.1f} minutes)"
        )

        if slowest_time >= 60:

            print(
                f"    Slowest replier : "
                f"{slowest_person} "
                f"(avg {slowest_time / 60:.1f} hours)"
            )

        else:

            print(
                f"    Slowest replier : "
                f"{slowest_person} "
                f"(avg {slowest_time:.1f} minutes)"
            )

    else:

        print(
            "    Not enough response data."
        )

    # ========================================================
    # SILENT STREAKS
    # ========================================================

    print()
    print("  LONGEST SILENT STREAKS")

    sorted_silent = sorted(
        silent_streaks.items(),
        key=lambda item: item[1]["days"],
        reverse=True
    )

    for person, data in sorted_silent[:5]:

        if data["days"] > 0:

            print(
                f"    {person[:18]:<18}: "
                f"{data['days']} days "
                f"("
                f"{data['start'].strftime('%d %b')}"
                f" - "
                f"{data['end'].strftime('%d %b')}"
                f")"
            )

    # ========================================================
    # PERSONALITY ARCHETYPES
    # ========================================================

    print()
    print("  PERSONALITY ARCHETYPES")

    display_archetypes(
        final_archetypes,
        participants
    )

    # ========================================================
    # FOOTER
    # ========================================================

    print()
    print("=" * 60)
    print(
        "Generated by GroupDNA  •  "
        "Built with Python + NumPy"
    )
    print("=" * 60)

In [ ]:
final_report()


GROUPDNA REPORT — "RVITM BATCH 7"
383 days  •  1,094 messages  •  179 participants

  Period          : 02 September 2025 to 19 September 2026
  Busiest day     : 31 May 2026 (138 messages)
  Busiest hour    : 23:00 - 00:00

  MESSAGES PER PERSON
    +91 77606 39981    ██████████████████████     63 (5.8%)
    +91 70199 98516    ███████████████████        57 (5.2%)
    +91 97437 43618    ███████████████            45 (4.1%)
    +91 63613 63701    ████████████               36 (3.3%)
    +91 90199 98404    ███████████                34 (3.1%)
    +91 78922 56796    ███████████                32 (2.9%)
    +91 91106 25515    ██████████                 30 (2.7%)
    +91 82178 99526    █████████                  28 (2.6%)
    +91 81471 73550    █████████                  26 (2.4%)
    +91 77804 60892    ████████                   25 (2.3%)
    +91 95699 65462    ███████                    22 (2.0%)
    +91 94485 42544    ███████                    21 (1.9%)
    +91 86606 01436    ██████   